In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load variables from .env file
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    os.environ["GROQ_API_KEY"] = groq_api_key

model = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=groq_api_key)

# Middleware
- Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following.

 - Tracking agent behavior with logging analysis and debugging ###
 - Transformation prompts, tool selection and output formating ###
 - Adding retries, fallback and early termination logic ###
 - Applying rate limits, guardrails and PII detection ###

## Summarization Middleware
automatically summarize conversation history when approaching token limits, preserving recent message while compressing older context summarization is used for the following:
- Long-running conversation that exceed context windows.
- Multi-turn dialogues with extensive history.
- Application where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

## agent 
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [ ]:
# Run with thread id 
config ={"configurable":{"thread_id":"test-1"}}

In [ ]:
questions = [
    "what is 2+2",
    "what is blue whale",
    "what is 8*8",
    "what is 4*4"
]
for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

questions = [
    "what is 2+2",
    "what is blue whale",
    "what is 8*8",
    "what is 4*4"
]
for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

## Token size

In [30]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

# Load variables from .env file
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long response to use more tokens """
    return f"""Hotel in {city}:
    1. Grand hotel - 5 star, $350/night, spa, pool, gym
    2. City inn - 4 star, $180/night, business center
    3. Budget stay - 3 star, $75/night, free wifi """

# 1. Instantiate the model object using ChatGroq
model = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=groq_api_key)

# 2. Token count function
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

# 3. Create the agent, passing the model and the custom token_counter
agent = create_agent(
    model=model,
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 550),
            keep=("tokens", 200),
            token_counter=count_tokens  # pass custom token counter here
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

In [31]:
# Run test
cities = ["Paris", "London", "Tokyo", "New york"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotel in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])}")
    print(f"{(response['messages'])}")

Paris: ~220 tokens, 8
[HumanMessage(content='Find hotel in Paris', additional_kwargs={}, response_metadata={}, id='87056ece-a534-4095-a4b2-a31621db0b10'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '51jxtv33b', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 226, 'total_tokens': 241, 'completion_time': 0.03877542, 'completion_tokens_details': None, 'prompt_time': 0.011664134, 'prompt_tokens_details': None, 'queue_time': 0.055212005, 'total_time': 0.050439554}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb355-8a71-7d11-be1d-77a865ff85f9-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'Paris'}, 'id': '51jxtv33b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_to

## based on Fraction

In [38]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~59 tokens (0.0461%), 6 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='a7b4ccad-d4b6-4f35-aa43-a912aaf96a1b'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2zbfw8cgz', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 210, 'total_tokens': 225, 'completion_time': 0.034127429, 'completion_tokens_details': None, 'prompt_time': 0.01416928, 'prompt_tokens_details': None, 'queue_time': 0.160836558, 'total_time': 0.048296709}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb35e-cdec-7763-8191-fa65f98bde66-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': '2zbfw8cgz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

## Human In the loop middleware
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [33]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [35]:
agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)


In [36]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [37]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='46205423-c190-4e64-b05d-295422ef599e'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5w4ee6jv9', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.050370838, 'completion_tokens_details': None, 'prompt_time': 0.030078984, 'prompt_tokens_details': None, 'queue_time': 0.055528915, 'total_time': 0.080449822}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb35e-5e8c-7011-9cad-f55a951553bd-0', tool_calls=[{'name': 'send_email_tool', 'args': {'b

In [39]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Here are the hotels in Singapore: 
1. Grand Hotel - $350
2. City Inn - $180
3. Budget Stay - $75


In [40]:
result

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to search for hotels in different cities.\n\n## SUMMARY\nThe user has searched for hotels in Paris, London, and is currently searching for hotels in Tokyo. The search results for Paris and London included three hotels: Grand Hotel, City Inn, and Budget Stay, with prices $350, $180, and $75 respectively.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe user's next step is to receive the search results for hotels in Tokyo. The AI should provide a list of hotels in Tokyo, along with their prices, in response to the user's query.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='d48c15dc-5cb0-4d6b-ba08-a55bee65adf0'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3rrhefq3e', 'function': {'arguments': '{"city":"Tokyo"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tok

## Reject

In [41]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# email read tool 
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

# email sender tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [42]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [43]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: 


In [44]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='01a166f8-9e5c-4cd8-8c1a-d4aaffb76226'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'scf2wg05g', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.050917077, 'completion_tokens_details': None, 'prompt_time': 0.015635641, 'prompt_tokens_details': None, 'queue_time': 0.164947289, 'total_time': 0.066552718}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb362-f207-7aa3-902d-83206945ef3c-0', tool_calls=[{'name': 'send_email_tool', 'args': {'b

## Editing

In [45]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)
config = {"configurable": {"thread_id": "test-edit"}}


In [46]:

config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [47]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='770dfb90-d95d-4234-8778-723c0a60f951'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ar814fr2p', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 307, 'total_tokens': 337, 'completion_time': 0.059205939, 'completion_tokens_details': None, 'prompt_time': 0.030100951, 'prompt_tokens_details': None, 'queue_time': 0.053949774, 'total_time': 0.08930689}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eb364-1aff-7eb2-bfa4-1acc7653c96c-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello'